# Build the face training set on Colab (GPU)

Runtime: **Runtime > Change runtime type > T4 GPU**.

Drive layout expected (`MyDrive/faceid/`):
```
faceid/
  faceid-adapter-code.zip           # the code (git archive)
  models/AuraFace-v1/               # AuraFace pack (parent folder must be named 'models')
  raw/ffhq512/*.parquet             # 54 shards, 27.1 GB   (optional per run)
  raw/celebamask/CelebAMask-HQ.zip  # 3.15 GB              (optional per run)
```
Each source is used if present, so it can be built in parts: CelebA-HQ first, FFHQ in a later session.
A previous `faces_built.zip` on Drive is restored first and the build **appends** to it (finished images are skipped).
Output: `MyDrive/faceid/faces_built.zip` (~9 GB when complete): unzip it as `data/faces` on the GPU box.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/faceid'
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
# Copy to the local disk first: reading many files from Drive is slow. Missing sources are skipped.
import os, glob
!mkdir -p /content/work/models /content/work/raw
!unzip -q -o "$D/faceid-adapter-code.zip" -d /content/work/code
!cp -r "$D/models/AuraFace-v1" /content/work/models/
if glob.glob(f'{D}/raw/ffhq512/*.parquet'):
    !cp -r "$D/raw/ffhq512" /content/work/raw/
if os.path.exists(f'{D}/raw/celebamask/CelebAMask-HQ.zip'):
    !unzip -q -o "$D/raw/celebamask/CelebAMask-HQ.zip" "CelebAMask-HQ/CelebA-HQ-img/*" -d /content/work/raw/
# restore an earlier partial build, so this run appends to it
if os.path.exists(f'{D}/faces_built.zip'):
    !unzip -q -o "$D/faces_built.zip" -d /content/work/
print('ffhq shards:', len(glob.glob('/content/work/raw/ffhq512/*.parquet')))
print('celeba images:', len(glob.glob('/content/work/raw/CelebAMask-HQ/CelebA-HQ-img/*.jpg')))
print('already built:', sum(1 for _ in open('/content/work/faces/meta.jsonl')) if os.path.exists('/content/work/faces/meta.jsonl') else 0)

In [ ]:
# insightface (AuraFace runtime) + AuraFace on CUDA + parquet reader.
# onnxruntime-gpu is pinned: the newest build targets CUDA 13, Colab has CUDA 12
# (it would silently fall back to CPU).
!pip -q install insightface==0.7.3 onnxruntime-gpu==1.22.0 pyarrow pyyaml
# a real CUDA session, not just the provider list
!python -c "import onnxruntime as ort; s = ort.InferenceSession('/content/work/models/AuraFace-v1/scrfd_10g_bnkps.onnx', providers=['CUDAExecutionProvider']); print('AuraFace session on:', s.get_providers()[0])"

In [ ]:
# The build: only the sources present are passed. Resumable and additive.
%cd /content/work/code
args = ['--gpu', '--aura', '/content/work/models/AuraFace-v1', '--out', '/content/work/faces']
if glob.glob('/content/work/raw/ffhq512/*.parquet'):
    args += ['--parquet', '/content/work/raw/ffhq512/*.parquet']
if os.path.isdir('/content/work/raw/CelebAMask-HQ/CelebA-HQ-img'):
    args += ['--images', '/content/work/raw/CelebAMask-HQ/CelebA-HQ-img']
cmd = ' '.join(f'"{a}"' for a in args)
!python prep/build_dataset.py {cmd}

In [ ]:
# Check, then save ONE zip to Drive (100k small files straight to Drive is very slow)
import json, numpy as np
meta = [json.loads(l) for l in open('/content/work/faces/meta.jsonl')]
emb = np.load('/content/work/faces/embeds.npy')
print(len(meta), 'faces |', emb.shape, '| val', sum(m['split'] == 'val' for m in meta))
assert len(meta) == len(emb)
!cd /content/work && zip -q -r -0 faces_built.zip faces && cp faces_built.zip "$D/" && ls -la "$D/faces_built.zip"